# 1. Setup & Load data

In [22]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Theme for the whole chart
pio.templates.default = "plotly_white"
COLORS = {"Adelie Penguin (Pygoscelis adeliae)": '#FF8C00', "Chinstrap penguin (Pygoscelis antarctica)": '#9932CC', "Gentoo penguin (Pygoscelis papua)": '#057076'}

In [23]:
# Loading penguins_clean.csv
df = pd.read_csv('/Users/anngothesoloist/Project_1_Data_Visualization_Penguins_Dashboard/data/ penguins_clean.csv')

# Check data
print(f"Cleaned dataset has: {df.shape[0]} rows and {df.shape[1]} columns")
display(df.describe())

Cleaned dataset has: 34 rows and 7 columns


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
count,34.000000,34.000000,34.000000,34.000000
mean,44.711765,17.647059,196.735294,3877.205882
std,5.800667,1.867138,12.473751,747.327756
min,35.900000,13.700000,172.000000,2700.000000
25%,39.325000,16.600000,190.000000,3343.750000
50%,44.500000,17.900000,195.500000,3737.500000
75%,49.600000,19.150000,201.500000,4237.500000
max,58.000000,20.000000,225.000000,5700.000000


# 2. EDA Visualizations (Directly run Notebook first)

In [24]:
# 1. Distribution: Weight by Species
fig1 = px.violin(df, y="body_mass_g", x="species", color="species", 
                 box=True, points="all", hover_data=df.columns,
                 title="Distribution of weights by species",
                 color_discrete_map=COLORS)
fig1.show()

In [25]:
# 2. Correlation: Correlation matrix of physical figures
corr = df.select_dtypes(include=[np.number]).corr()
fig2 = px.imshow(corr, text_auto=True, aspect="auto",
                 title="Correlation matrix of physical figures",
                 color_continuous_scale='RdBu_r')
fig2.show()

In [26]:
# 3. Scatter Matrix: Multivariable scatter matrix
# Scatter Matrix with horizontal Y-labels
fig3 = px.scatter_matrix(
    df, 
    dimensions=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"],
    color="species", 
    color_discrete_map=COLORS,
    title="Penguin Morphological Matrix",
    # Replacing "_"
    labels={col: col.replace('_', ' ').title() for col in df.columns}
)

fig3.update_traces(diagonal_visible=True)

# Modify y-axis for var names
fig3.update_yaxes(
    tickangle=0, 
    title_standoff=10, 
    automargin=True   
)

# Overall layout
fig3.update_layout(
    height=900, 
    width=1200,
    margin=dict(l=180, r=20, t=80, b=80), 
    autosize=True,
    title={
        'text': "Penguin Morphological Matrix",
        'y': 0.95,           
        'x': 0.5,            
        'xanchor': 'center', 
        'yanchor': 'top'
    }
)

fig3.show()

In [27]:
# 4. Species Count Bar Chart
# This visualization shows the sample size for each species in the dataset.
fig4 = px.histogram(
    df, 
    x="species", 
    color="species",
    color_discrete_map=COLORS,
    title="Total Count of Each Penguin Species",
    text_auto=True # Automatically displays the count
)

# Refine the layout and appearance
fig4.update_layout(
    title_x=0.5,
    xaxis_title="Penguin Species",
    yaxis_title="Number of Individuals",
    showlegend=False,          # Hide legend as X-axis already has the names
    plot_bgcolor="white",      # Clean white background
    bargap=0.3,                # Add space between bars (0.0 to 1.0)
    height=500
)

# Fine-tune the text labels and axis lines
fig4.update_traces(
    textposition='outside',    # Move numbers to the top of the bars
    textfont_size=14,
    marker_line_color='black', # Add a subtle border to bars
    marker_line_width=1
)

# Optional: Add subtle horizontal gridlines for easier reading
fig4.update_yaxes(showgrid=True, gridcolor='LightGray')

fig4.show()

In [28]:
# 5. Scatter Plot with Marginal Histograms
# Analyzing the relationship between Flipper Length and Bill Length with distribution density
fig5 = px.scatter(
    df, 
    x="flipper_length_mm", 
    y="bill_length_mm", 
    color="species",
    marginal_x="histogram", # Shows distribution on top
    marginal_y="violin",       # Shows box plot on the right
    color_discrete_map=COLORS,
    trendline="ols",        # Optional: Adds a linear regression line for each species
    title="Flipper Length vs. Bill Length (with Marginal Distributions)",
    labels={
        "flipper_length_mm": "Flipper Length (mm)",
        "bill_length_mm": "Bill Length (mm)"
    }
)

fig5.update_layout(
    title_x=0.5,
    height=600
)
fig5.show()

In [29]:
# 6. Horizontal Facetted Bar Chart: Average Body Mass
# Swapping X and Y to make bars horizontal for easier label reading

# Grouping the data
df_grouped = df.groupby(['island', 'species', 'sex'])['body_mass_g'].mean().reset_index()

# 6. Horizontal Bar Chart
fig6_horizontal = px.bar(
    df_grouped,
    x="body_mass_g",          # Numerical value now on X-axis
    y="species",              # Categories now on Y-axis
    color="sex",              
    orientation='h',          # Set orientation to horizontal
    barmode="group",          
    facet_col="island",       # Keep islands in separate columns
    color_discrete_map={      
        "Male": "#7f7f7f",    
        "Female": "#bcbd22"   
    },
    title="Average Body Mass by Species, Island, and Sex",
    labels={
        "species": "Species",
        "body_mass_g": "Avg Body Mass (g)",
        "sex": "Sex",
        "island": "Island"
    }
)

fig6_horizontal.update_yaxes(
    tickangle=0,
    ticksuffix="   ",
    griddash='dot'
)

fig6_horizontal.update_layout(
    title_x=0.5,
    height=500,
    margin=dict(t=80, l=100, r=40, b=80), # Increase left margin for species names
    legend=dict(orientation="h", yanchor="bottom", y=-0.4, xanchor="center", x=0.5)
)

# Adjust text labels to appear to the right of the horizontal bars
fig6_horizontal.update_traces(
    texttemplate='%{x:.0f}', 
    textposition='outside'
)

fig6_horizontal.show()

In [30]:
# 7. Donut Chart: Species Proportion per Island
# This shows how "exclusive" certain islands are to specific species.
fig7 = px.pie(
    df, 
    names="species", 
    facet_col="island",
    color="species",
    color_discrete_map=COLORS,
    hole=0.4, # Creates the "Donut" look
    title="Species Composition by Island"
)

fig7.update_layout(title_x=0.5)
fig7.show()

In [31]:
# 8. Enhanced 3D Scatter Plot (Shifted Upwards)
# This increases the frame size and moves the 3D scene higher within the layout
fig8 = px.scatter_3d(
    df, x='bill_length_mm', y='flipper_length_mm', z='body_mass_g',
    color='species', color_discrete_map=COLORS,
    opacity=0.8, title="3D Morphological Clusters",
    labels={"bill_length_mm": "Bill", "flipper_length_mm": "Flipper", "body_mass_g": "Mass"}
)

fig8.update_traces(marker=dict(size=4, line=dict(width=1, color='DarkSlateGrey')))

fig8.update_layout(
    height=950,      
    width=1100,         
    title_x=0.5,
    title_font_size=22,
    
    # The 'scene' controls the 3D c
    scene=dict(
        # domain y: [0.2, 1.0] pushes the cube 20% up
        domain=dict(y=[0.2, 1.0]), 
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
        xaxis=dict(gridcolor='white', zerolinecolor='white', showbackground=True, backgroundcolor="rgb(230, 230, 230)"),
        yaxis=dict(gridcolor='white', zerolinecolor='white', showbackground=True, backgroundcolor="rgb(230, 230, 230)"),
        zaxis=dict(gridcolor='white', zerolinecolor='white', showbackground=True, backgroundcolor="rgb(230, 230, 230)")
    ),
    
    # Adjusting margins
    margin=dict(l=0, r=0, b=50, t=100),
    
    # Legend placement
    legend=dict(
        yanchor="top",
        y=0.9,
        xanchor="center",
        x=0.05,
        bgcolor="rgba(255, 255, 255, 0.5)"
    )
)

fig8.show()